# 05 AGENT HIERARCHY

> 核心：多智能体系统的组织方式决定了系统的上限。单一 Agent 无法同时具备领域专业性和全局协调能力。

## 5.1 为什么需要 Agent 层级？

### 单一 Agent 的瓶颈

- **上下文溢出**：复杂任务需要太多信息，单一 Agent 的上下文窗口有限
- **能力冲突**：需要同时是领域专家和协调者，角色冲突
- **错误级联**：一个 Agent 的错误会级联传播到整个系统
- **无法并行**：所有任务串行执行，效率低下

### 层级架构的优势

| 维度 | 单一 Agent | 层级架构 |
|------|-----------|----------|
| 上下文 | 受限 | 按层分配，互不干扰 |
| 角色 | 混合（专家+协调） | 分离（专家 ≠ 协调） |
| 错误传播 | 级联 | 隔离 |
| 并行性 | 差 | 好（层内可并行） |

In [ ]:
# Agent 层级架构模拟
class Agent:
    def __init__(self, name, role, capabilities):
        self.name = name
        self.role = role
        self.capabilities = capabilities

    def process(self, task):
        print(f"[{self.name}] 处理任务: {task[:30]}...")
        return {"status": "done", "agent": self.name}

# 层级 1: 协调者 Agent（Manager）
manager = Agent("Manager", "coordinator", ["分解任务", "分配任务", "汇总结果"])

# 层级 2: 领域专家 Agent（Specialists）
frontend_expert = Agent("Frontend-Expert", "ui_specialist", ["React", "CSS", "A11y"])
backend_expert = Agent("Backend-Expert", "api_specialist", ["Python", "PostgreSQL", "Redis"])
data_expert = Agent("Data-Expert", "data_specialist", ["Pandas", "SQL", "Analytics"])

# 模拟任务分解
task = "构建一个用户分析仪表盘，包含实时活跃用户数和历史趋势图"
print(f"原始任务: {task}")
print(f"\n[{manager.name}] 将任务分解为子任务")
print(f"  - 子任务1: 前端仪表盘 UI (分配给 {frontend_expert.name})")
print(f"  - 子任务2: 后端 API (分配给 {backend_expert.name})")
print(f"  - 子任务3: 数据查询逻辑 (分配给 {data_expert.name})")

## 5.2 三层 Agent 架构

### 推荐架构：Manager → Specialist → Executor

```
┌─────────────────────────────────────────────────────────┐
│                    Manager Agent                        │
│         (全局协调、任务分解、结果汇总)                   │
├──────────────┬──────────────┬───────────────────────────┤
│ Specialist A │ Specialist B │ Specialist C              │
│  (领域专家)   │  (领域专家)   │  (领域专家)               │
├──────────────┴──────────────┴───────────────────────────┤
│                   Executor Agents                       │
│              (具体执行工具调用)                          │
└─────────────────────────────────────────────────────────┘
```

### 各层职责

- **Manager**：理解任务全局、分解为子任务、分配给合适的 Specialist、汇总结果
- **Specialist**：深度理解领域知识、检查子任务是否合规、调用 Executor
- **Executor**：执行具体操作（搜索、代码生成、文件操作）

In [ ]:
# Manager Agent 的决策逻辑模拟
def manager_decision(task_description):
    task_keywords = {
        "frontend": ["界面", "UI", "按钮", "布局", "样式", "前端"],
        "backend": ["API", "数据库", "后端", "服务", "接口"],
        "data": ["分析", "统计", "报表", "图表", "数据"],
    }

    detected = {}
    for domain, keywords in task_keywords.items():
        matches = sum(1 for kw in keywords if kw in task_description)
        if matches > 0:
            detected[domain] = matches

    if detected:
        top_domain = max(detected, key=detected.get)
        print(f"Manager 判断: 任务主要涉及 {top_domain} 领域")
        print(f"  置信度: {detected[top_domain] / sum(detected.values()):.0%}")
        return top_domain
    return "unknown"

task1 = "构建一个用户分析仪表盘"
task2 = "实现用户登录的 API 接口"

manager_decision(task1)
manager_decision(task2)

## 5.3 跨层通信协议

### 信息传递的规范

```
Manager → Specialist: 任务描述 + 约束条件 + 输出格式
Specialist → Manager: 执行结果 + 问题描述 + 风险提示
Specialist → Executor: 具体操作指令
Executor → Specialist: 操作结果
```

### 消息格式示例

```json
{
  "from": "manager",
  "to": "frontend-specialist",
  "task": "实现用户列表组件",
  "constraints": [
    "使用 React 18",
    "遵循 Design System 的颜色令牌",
    "支持响应式布局"
  ],
  "output_format": "React Component Code"
}
```

In [ ]:
# Specialist 向 Manager 报告的格式
def specialist_report(success, result=None, issues=None, risks=None):
    return {
        "success": success,
        "result": result,
        "issues": issues or [],
        "risks": risks or [],
        "next_action": "continue" if success else "need_human_input"
    }

# 示例报告
report1 = specialist_report(
    success=True,
    result="React 组件已生成，包含 3 个子组件",
    issues=[],
    risks=["可能需要调整响应式断点"]
)
print("报告示例:", report1)

## 5.4 层级架构的实践指南

### 何时使用层级架构

| 任务复杂度 | Agent 类型 | 说明 |
|-----------|-----------|------|
| 低（简单查询） | 单一 Agent | 不需要层级 |
| 中（多步骤） | Agent + Tools | 单层足够 |
| 高（多领域） | Manager + Specialist | 需要层级 |
| 极高（跨系统） | 多 Manager 协作 | 需要更复杂架构 |

### 避免过度设计

- **教训**：很多场景不需要复杂的层级架构
- **原则**：先让单一 Agent 工作，再按需拆分
- **信号**：当单一 Agent 出现上下文溢出或角色冲突时，再引入层级

In [ ]:
# 层级架构的适用性判断
def should_use_hierarchy(task):
    complexity_indicators = [
        "多个领域", "跨系统", "多人协作",
        "需要分解", "子任务", "并行"
    ]

    score = sum(1 for indicator in complexity_indicators if indicator in task)

    if score >= 2:
        print(f"任务复杂度评分: {score}/6 → 建议使用层级架构")
        return True
    else:
        print(f"任务复杂度评分: {score}/6 → 单一 Agent 足够")
        return False

should_use_hierarchy("构建一个跨多个系统的数据分析平台")
should_use_hierarchy("查询今天的天气")

## 5.5 实验结论

### Agent 层级架构的核心价值
- **上下文隔离**：每层只关注自己的领域知识
- **角色分离**：协调者不干预专业决策，专家不关心全局协调
- **并行效率**：层内 Specialist 可并行工作
- **错误隔离**：单点错误不会级联传播

### 关键设计原则
- **按需分层**：不要过度设计，从单一 Agent 开始
- **清晰协议**：层间通信必须有明确的消息格式
- **单点决策**：Manager 负责全局决策，避免多头决策
- **可观测性**：每层的输入输出必须可追踪